# Rank × hidden-size sweep — results analysis

Loads the aggregated CSV produced by `scripts/19_05_26_sweep_rank_representation.py`
and visualises performance and representation metrics across the rank × hidden-size grid.

**Note:** Training curves were not saved in this sweep run.  Only final inference
metrics are available.  To add training-curve tracking, run the sweep again after
adding `--save-curves` support.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

from cxval.vis import STYLE
plt.style.use(STYLE)

In [ ]:
# ── set path to the CSV ──────────────────────────────────────────────────────
SWEEP_DIR = Path("../results/19_05_26_sweep_rank_representation")

csvs = sorted(SWEEP_DIR.glob("rank_sweep_*.csv"))
if not csvs:
    raise FileNotFoundError(f"No rank_sweep_*.csv found in {SWEEP_DIR}")
CSV_PATH = csvs[-1]   # most recent

df = pd.read_csv(CSV_PATH)
print(f"Loaded: {CSV_PATH}")
print(f"Shape:  {df.shape}")
print(df.head())

# sentinel: rank=None stored as -1 in CSV
FULL_RANK_KEY = -1

hidden_sizes = sorted(df["hidden_size"].unique().tolist())
rank_keys    = sorted(r for r in df["rank_key"].unique() if r != FULL_RANK_KEY)
n_seeds      = df.groupby(["hidden_size", "rank_key"]).size().max()

print(f"\nhidden_sizes: {hidden_sizes}")
print(f"rank values:  {rank_keys}  (plus full-rank, key={FULL_RANK_KEY})")
print(f"seeds per config (max): {n_seeds}")

In [ ]:
# ── compute PSA score from existing columns ───────────────────────────────────
# psa_score = 1 - mean(|lick_high - 1| + |lick_low - 0|) / 2
#           = (lick_high + (1 - lick_low)) / 2
# psa_delta = lick_high - lick_low
df["psa_score"] = (df["lick_rate_high"] + (1.0 - df["lick_rate_low"])) / 2.0
df["psa_delta"] = df["lick_rate_high"] - df["lick_rate_low"]

print("PSA columns added.")
print(df[["hidden_size", "rank_key", "psa_score", "psa_delta"]].head(12))

## Performance metrics — heatmaps and line plots

In [ ]:
# Helper: mean ± std for (hidden_size, rank_key)
def _stats(col, h, rk):
    vals = df.loc[(df["hidden_size"] == h) & (df["rank_key"] == rk), col].dropna()
    return (float(vals.mean()), float(vals.std())) if len(vals) else (np.nan, np.nan)


def heatmap_metric(ax, col, h_sizes, r_keys, vmin, vmax, cmap, title, fmt=".2f",
                   annot_fn=None):
    """Plot mean of `col` as a heatmap (rows=hidden_size, cols=rank or 'full')."""
    all_rk = r_keys + [FULL_RANK_KEY]
    data   = np.full((len(h_sizes), len(all_rk)), np.nan)
    for ri, rk in enumerate(all_rk):
        for hi, h in enumerate(h_sizes):
            mu, _ = _stats(col, h, rk)
            data[hi, ri] = mu

    im = ax.imshow(data, vmin=vmin, vmax=vmax, cmap=cmap, aspect="auto")
    for hi in range(len(h_sizes)):
        for ri in range(len(all_rk)):
            v = data[hi, ri]
            if not np.isnan(v):
                label = annot_fn(v) if annot_fn else f"{v:{fmt}}"
                ax.text(ri, hi, label, ha="center", va="center", fontsize=7)

    xlbls = [str(r) for r in r_keys] + ["full"]
    ax.set_xticks(range(len(all_rk)))
    ax.set_xticklabels(xlbls, fontsize=8)
    ax.set_yticks(range(len(h_sizes)))
    ax.set_yticklabels([str(h) for h in h_sizes], fontsize=8)
    ax.set_xlabel("Rank", fontsize=8)
    ax.set_ylabel("Hidden size", fontsize=8)
    ax.set_title(title, fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
    return im


# lick_rate_mid is only present in CSVs produced after adding it to the sweep script
_has_mid = "lick_rate_mid" in df.columns

PERF_METRICS = [
    ("spearman_r",          "Lick–value calibration\n(Spearman r, ↑→1)",        -1, 1,   "RdYlGn"),
    ("psa_score",           "PSA score\n(1−MAE, ↑→1)",                            0, 1,   "RdYlGn"),
    ("psa_delta",           "PSA delta\n(high−low, ↑→1)",                         0, 1,   "RdYlGn"),
    ("lick_rate_high",      "Lick rate — high stim\n(↑→1)",                       0, 1,   "RdYlGn"),
    ("lick_rate_mid",       "Lick rate — mid stim\n(~0.5 expected)",               0, 1,   "RdYlGn"),
    ("lick_rate_low",       "Lick rate — low stim\n(↓→0)",                        0, 1,   "RdYlGn_r"),
    ("pct_reward_consumed", "Reward consumed (%)\n(↑ better)",                    0, 100, "RdYlGn"),
    ("false_alarm_rate",    "False alarm rate (%)\n(↓ better)",                   0, 100, "RdYlGn_r"),
]

# Drop mid-stim entry if not in this CSV
if not _has_mid:
    print("Note: lick_rate_mid not in this CSV — re-run the sweep to populate it.")
    PERF_METRICS = [m for m in PERF_METRICS if m[0] != "lick_rate_mid"]

n_perf = len(PERF_METRICS)
ncols  = 4
nrows  = int(np.ceil(n_perf / ncols)) + (1 if n_perf % ncols == 0 else 0)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
for ax, (col, title, vmin, vmax, cmap) in zip(axes.flat, PERF_METRICS):
    heatmap_metric(ax, col, hidden_sizes, rank_keys, vmin, vmax, cmap, title)

for ax in axes.flat[n_perf:]:
    ax.axis("off")

fig.suptitle(
    f"Performance metrics — mean over {n_seeds} seeds\n"
    f"rows = hidden size, cols = rank (rightmost = full-rank baseline)",
    y=1.02, fontsize=10,
)
plt.tight_layout()
plt.show()

## Representation metrics — heatmaps

In [ ]:
REPR_METRICS = [
    ("held_out_r",               "Held-out decode\n(train=anchor → test=swap, Pearson r)",  -1, 1,  "RdBu"),
    ("held_out_r_swap_to_anchor","Held-out decode\n(train=swap → test=anchor, Pearson r)",  -1, 1,  "RdBu"),
    ("ccgp_ridge_anchor",        "CCGP Ridge\n(train=all, test=anchor)",                    -1, 1,  "RdBu"),
    ("ccgp_ridge_swap",          "CCGP Ridge\n(train=all, test=swap)",                      -1, 1,  "RdBu"),
    ("ccgp_bin_anchor",          "CCGP SVM\n(train=all, test=anchor)",                       0, 1,  "RdBu"),
    ("ccgp_bin_swap",            "CCGP SVM\n(train=all, test=swap)",                         0, 1,  "RdBu"),
    ("rsa_identity_r",           "RSA — identity model\n(Spearman r)",                      -1, 1,  "RdBu"),
    ("rsa_value_r",              "RSA — value model\n(Spearman r)",                         -1, 1,  "RdBu"),
]

# Drop metrics missing from this CSV (e.g. held_out_r_swap_to_anchor from older runs)
REPR_METRICS = [m for m in REPR_METRICS if m[0] in df.columns or m[0] in ("held_out_r", "ccgp_ridge_anchor", "ccgp_ridge_swap", "ccgp_bin_anchor", "ccgp_bin_swap", "rsa_identity_r", "rsa_value_r")]
REPR_METRICS = [m for m in REPR_METRICS if m[0] in df.columns]

n_repr = len(REPR_METRICS)
ncols  = 4
nrows  = int(np.ceil(n_repr / ncols)) + (1 if n_repr % ncols == 0 else 0)
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
for ax, (col, title, vmin, vmax, cmap) in zip(axes.flat, REPR_METRICS):
    heatmap_metric(ax, col, hidden_sizes, rank_keys, vmin, vmax, cmap, title)

for ax in axes.flat[n_repr:]:
    ax.axis("off")

fig.suptitle(
    f"Representation metrics — mean over {n_seeds} seeds\n"
    f"rows = hidden size, cols = rank (rightmost = full-rank baseline)",
    y=1.02, fontsize=10,
)
plt.tight_layout()
plt.show()

## Line plots — metric vs rank, coloured by hidden size (with error bars)

In [ ]:
h_palette = plt.cm.plasma(np.linspace(0.15, 0.85, len(hidden_sizes)))
h_color   = {h: c for h, c in zip(hidden_sizes, h_palette)}

all_rk_keys = rank_keys + [FULL_RANK_KEY]
x_pos  = list(range(len(all_rk_keys)))
x_lbls = [str(r) for r in rank_keys] + ["full"]

ALL_METRICS = [
    ("spearman_r",                 "Lick–value calibration (Spearman r)",         -1, 1,  None),
    ("psa_score",                  "PSA score  (1−MAE)",                            0, 1,  None),
    ("psa_delta",                  "PSA delta  (high−low)",                         0, 1,  None),
    ("lick_rate_high",             "Lick rate — high stim",                         0, 1,  1.0),
    ("lick_rate_mid",              "Lick rate — mid stim",                          0, 1,  0.5),
    ("lick_rate_low",              "Lick rate — low stim",                          0, 1,  0.0),
    ("held_out_r",                 "Held-out decode  (anchor→swap)",               -1, 1,  0.0),
    ("held_out_r_swap_to_anchor",  "Held-out decode  (swap→anchor)",               -1, 1,  0.0),
    ("ccgp_ridge_anchor",          "CCGP Ridge — anchor",                          -1, 1,  0.0),
    ("ccgp_ridge_swap",            "CCGP Ridge — swap",                            -1, 1,  0.0),
    ("ccgp_bin_anchor",            "CCGP SVM — anchor",                             0, 1,  0.5),
    ("ccgp_bin_swap",              "CCGP SVM — swap",                               0, 1,  0.5),
    ("rsa_identity_r",             "RSA — identity model",                         -1, 1,  0.0),
    ("rsa_value_r",                "RSA — value model",                            -1, 1,  0.0),
]

# Drop columns not present in this CSV
ALL_METRICS = [m for m in ALL_METRICS if m[0] in df.columns]

n_met  = len(ALL_METRICS)
ncols  = 4
nrows  = int(np.ceil((n_met + 1) / ncols))   # +1 for legend panel
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), constrained_layout=True)

for ax, (col, title, ymin, ymax, ref) in zip(axes.flat, ALL_METRICS):
    for h in hidden_sizes:
        ys, es = [], []
        for rk in all_rk_keys:
            mu, sd = _stats(col, h, rk)
            ys.append(mu)
            es.append(sd if not np.isnan(sd) else 0.0)
        ax.errorbar(x_pos[:-1], ys[:-1], yerr=es[:-1],
                    color=h_color[h], marker="o", lw=2, capsize=3, label=f"h={h}")
        if not np.isnan(ys[-1]):
            ax.errorbar([x_pos[-1]], [ys[-1]], yerr=[es[-1]],
                        color=h_color[h], marker="D", ms=8, lw=1.5,
                        linestyle=":", capsize=3)

    if ref is not None:
        ax.axhline(ref, color="gray", lw=0.8, linestyle="--", alpha=0.6)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_lbls, fontsize=8)
    ax.set_xlim(-0.5, len(x_pos) - 0.5)
    ax.set_ylim(ymin - 0.05 * (ymax - ymin), ymax + 0.05 * (ymax - ymin))
    ax.set_xlabel("Rank")
    ax.set_title(title, fontsize=9)

# Legend panel
from matplotlib.lines import Line2D
leg_ax = list(axes.flat)[n_met]
leg_ax.axis("off")
leg_h = [Line2D([0],[0], color=h_color[h], lw=2, marker="o", label=f"h={h} (low-rank)")
         for h in hidden_sizes]
leg_h += [Line2D([0],[0], color=h_color[h], lw=1.5, linestyle=":", marker="D", label=f"h={h} (full-rank)")
          for h in hidden_sizes]
leg_ax.legend(handles=leg_h, fontsize=8, loc="center", frameon=False, ncol=2)

for ax in list(axes.flat)[n_met + 1:]:
    ax.axis("off")

plt.suptitle(
    f"All metrics vs rank  (n={n_seeds} seeds)  —  error bars = ±1 SD",
    y=1.01, fontsize=11,
)
plt.show()

## Diverged runs

In [ ]:
n_div = df["diverged"].sum() if "diverged" in df.columns else 0
print(f"Diverged runs: {n_div} / {len(df)}")
if n_div:
    print(df[df["diverged"]][[["hidden_size","rank_key","seed"]]])

# fraction diverged per config
if "diverged" in df.columns:
    div_frac = (df.groupby(["hidden_size", "rank_key"])["diverged"]
                  .mean()
                  .unstack("rank_key")
                  .rename(columns={FULL_RANK_KEY: "full"}))
    print("\nDivergence fraction per config:")
    print(div_frac.to_string(float_format="{:.2f}".format))